# Run inference for Qwen3TTS

This notebook demonstrates how to:
1. Initialize the Qwen3TTS model with dummy weights
2. Run inference with random embeddings to verify the model works


In [ ]:
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"


In [ ]:
import torch
from pathlib import Path
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

print("Imports successful")


In [ ]:
# Load vLLM engine with dummy model
type_str = "float32"
torch_type = getattr(torch, type_str)

max_len = 256
config_path = Path("dummy_qwen3_tts_model")
engine_args = AsyncEngineArgs(
    model=str(config_path.absolute()),
    dtype=type_str,
    max_model_len=max_len,
    max_num_batched_tokens=max_len,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using custom inputs
    enable_prefix_caching=False,
    #load_format="dummy",  # Use dummy loader for random weights
    trust_remote_code=True,
    #enforce_eager=True,
)

print("Initializing engine...")
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=max_len, skip_sampling=False)

print("Engine initialized successfully")


In [ ]:
# load prefill emb that contains text to generate and speaker info

prefill_data = torch.load("/home/vklimkov/workspace/vllm/vllm/dummy_qwen3_tts_model/prefill_input.pt")
prefill_emb = prefill_data["talker_input_embeds"][0].float().contiguous().cpu()
print(prefill_emb.shape)

In [ ]:

request_id = "test_request_1"


# Prepare inputs for prefill stage
prompt_len = prefill_emb.shape[0]

#prefill_emb = torch.randn_like(prefill_emb)
inputs = {
    # Dummy token IDs (required by vLLM, but we use custom_inputs for actual data)
    "prompt_token_ids": [0] * prompt_len,
    # Custom inputs for Qwen3TTS
    "custom_inputs": {
        "combined_embeddings": prefill_emb,
    }
}

print(f"Starting generation with request_id: {request_id}")
print(f"Prompt length: {prompt_len}")

generated_codecs = []
step_count = 0

async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
    # Extract generated codec tokens from custom outputs
    # The exact key depends on how Qwen3TTS returns outputs
    codec_tokens = output.outputs[0].custom_outputs["codes"]
    next_input = output.outputs[0].custom_outputs["next_input_embeddings"]
    generated_codecs.append(codec_tokens[-1:])
    step_count += 1
    print(f"Step {step_count}: Generated codecs shape: {codec_tokens.shape}, next_input shape: {next_input.shape}")
    
    # For testing, we'll generate a few steps then stop
    if step_count >= 30:
        await engine.abort(request_id)
        break

    new_custom_inputs = {
        "combined_embeddings": next_input[-1:, :],
    }
    await engine.append_request(request_id=request_id, custom_inputs=new_custom_inputs)

In [ ]:
arr = torch.cat(generated_codecs, dim=0)

import matplotlib.pyplot as plt

plt.imshow(arr.cpu().numpy().T, aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
print(arr.shape)
print(torch.min(arr), torch.max(arr))
torch.save(arr, "/home/vklimkov/workspace/qwen3_tts/Qwen3-TTS/vllm_pred_tokens.pt")